<a href="https://colab.research.google.com/github/Burakcelebi2003/MLB-Player-Game-Prediction-Platform/blob/main/MLB_Game_Results_Prediction_Platform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install streamlit numpy scipy pandas plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 47.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 75.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [streamlit]


In [ ]:
#Maç Analizi (ML)
%%writefile sabermetrics_gui.py
import streamlit as st
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.base import clone
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor
)
from sklearn.linear_model import Ridge, PoissonRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# Optional XGBoost ve SHAP
try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False

try:
    import shap
    import matplotlib.pyplot as plt
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False


# ============================================================
# 1. FEATURE ENGINEERING
# ============================================================

class MLFeatureExtractor:
    """
    Maç tahminine uyarlanmış özellik çıkarıcı.

    Hücum:
        xwOBA / wRC+ / OPS
        Season / L30 / L7

    Savunma:
        Starting Pitcher:
            xERA / xFIP
            Season / L30 / L7

        Bullpen:
            xERA / xFIP
            Season / L30 / L7
    """

    @staticmethod
    def extract_time_series_features(scores, prefix="runs"):
        arr = np.asarray(scores, dtype=float)

        if len(arr) == 0:
            arr = np.array([0.0])

        n = len(arr)
        x = np.arange(n)

        features = {
            f"{prefix}_mean": float(np.mean(arr)),
            f"{prefix}_std": float(np.std(arr, ddof=0)),
            f"{prefix}_max": float(np.max(arr)),
            f"{prefix}_min": float(np.min(arr)),
            f"{prefix}_median": float(np.median(arr)),

            # EWMA(3)
            f"{prefix}_ewma": float(
                pd.Series(arr)
                .ewm(span=min(3, n), adjust=False)
                .mean()
                .iloc[-1]
            ),
        }

        # Trend
        if n > 1:
            slope, _ = np.polyfit(x, arr, 1)

            features[f"{prefix}_slope"] = float(slope)
            features[f"{prefix}_recent_delta"] = float(
                arr[-1] - arr[0]
            )
        else:
            features[f"{prefix}_slope"] = 0.0
            features[f"{prefix}_recent_delta"] = 0.0

        # Rolling windows
        for window in [3, 7, 15, 30]:

            if n >= window:
                recent = arr[-window:]
            else:
                recent = arr

            features[f"{prefix}_l{window}_mean"] = float(
                np.mean(recent)
            )

            features[f"{prefix}_l{window}_std"] = float(
                np.std(recent, ddof=0)
            )

            features[f"{prefix}_l{window}_ewma"] = float(
                pd.Series(recent)
                .ewm(
                    span=min(3, len(recent)),
                    adjust=False
                )
                .mean()
                .iloc[-1]
            )

        features[f"{prefix}_last"] = float(arr[-1])

        return features

    # --------------------------------------------------------
    # OFFENSE
    # --------------------------------------------------------

    @staticmethod
    def extract_sabermetric_features(team_stats, league_stats):

        feats = {}

        metrics = [
            "xwOBA",
            "wRC+",
            "OPS"
        ]

        for metric in metrics:

            mean_val = float(
                league_stats[metric]["mean"]
            )

            std_val = float(
                league_stats[metric]["std"]
            )

            season = float(
                team_stats[metric]["season"]
            )

            l30 = float(
                team_stats[metric]["l30"]
            )

            l7 = float(
                team_stats[metric]["l7"]
            )

            for label, value in [
                ("season", season),
                ("l30", l30),
                ("l7", l7)
            ]:

                feats[
                    f"{metric}_z_{label}"
                ] = (
                    (value - mean_val) / std_val
                    if std_val > 0
                    else 0.0
                )

            feats[
                f"{metric}_delta_l7_season"
            ] = l7 - season

            feats[
                f"{metric}_delta_l30_season"
            ] = l30 - season

        return feats

    # --------------------------------------------------------
    # PITCHING
    # --------------------------------------------------------

    @staticmethod
    def extract_pitching_features(
        pitching_stats,
        prefix
    ):
        """
        prefix:
            sp
            bp

        Metrics:
            xERA
            xFIP

        Periods:
            season
            l30
            l7
        """

        feats = {}

        metrics = [
            "xERA",
            "xFIP"
        ]

        for metric in metrics:

            season = float(
                pitching_stats[metric]["season"]
            )

            l30 = float(
                pitching_stats[metric]["l30"]
            )

            l7 = float(
                pitching_stats[metric]["l7"]
            )

            for label, value in [
                ("season", season),
                ("l30", l30),
                ("l7", l7)
            ]:

                feats[
                    f"{prefix}_{metric.lower()}_{label}"
                ] = value

            feats[
                f"{prefix}_{metric.lower()}_delta_l7_season"
            ] = l7 - season

            feats[
                f"{prefix}_{metric.lower()}_delta_l30_season"
            ] = l30 - season

        return feats

    @staticmethod
    def add_pitching_context(
        features,
        context
    ):

        features.update({

            "opp_sp_xera_season":
                float(context.get(
                    "sp_xera_season", 4.0
                )),

            "opp_sp_xera_l30":
                float(context.get(
                    "sp_xera_l30", 4.0
                )),

            "opp_sp_xera_l7":
                float(context.get(
                    "sp_xera_l7", 4.0
                )),

            "opp_sp_xfip_season":
                float(context.get(
                    "sp_xfip_season", 4.0
                )),

            "opp_sp_xfip_l30":
                float(context.get(
                    "sp_xfip_l30", 4.0
                )),

            "opp_sp_xfip_l7":
                float(context.get(
                    "sp_xfip_l7", 4.0
                )),

            "opp_bp_xera_season":
                float(context.get(
                    "bp_xera_season", 4.2
                )),

            "opp_bp_xera_l30":
                float(context.get(
                    "bp_xera_l30", 4.2
                )),

            "opp_bp_xera_l7":
                float(context.get(
                    "bp_xera_l7", 4.2
                )),

            "opp_bp_xfip_season":
                float(context.get(
                    "bp_xfip_season", 4.2
                )),

            "opp_bp_xfip_l30":
                float(context.get(
                    "bp_xfip_l30", 4.2
                )),

            "opp_bp_xfip_l7":
                float(context.get(
                    "bp_xfip_l7", 4.2
                )),

            "home_advantage":
                float(context.get(
                    "home_advantage", 1
                )),

            "park_factor":
                float(context.get(
                    "park_factor", 100
                )),
        })

        return features


# ============================================================
# 2. MODEL FACTORY
# ============================================================

def model_factory(model_name):

    if model_name == "Random Forest":

        return RandomForestRegressor(
            n_estimators=300,
            max_depth=6,
            min_samples_leaf=4,
            random_state=42
        )

    if model_name == "Gradient Boosting":

        return GradientBoostingRegressor(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            loss="huber",
            random_state=42
        )

    if model_name == "HistGradient Boosting":

        return HistGradientBoostingRegressor(
            max_iter=250,
            learning_rate=0.05,
            l2_regularization=1.5,
            random_state=42
        )

    if model_name == "XGBoost" and XGB_AVAILABLE:

        return XGBRegressor(
            n_estimators=250,
            max_depth=3,
            learning_rate=0.05,
            objective="reg:squarederror",
            random_state=42
        )

    if model_name == "Poisson Regression":

        return Pipeline([
            (
                "scaler",
                StandardScaler()
            ),
            (
                "poisson",
                PoissonRegressor(
                    alpha=0.8,
                    max_iter=1000
                )
            )
        ])

    return Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "ridge",
            Ridge(alpha=5.0)
        )
    ])


# ============================================================
# 3. CLASSICAL FORECASTING MODELS
# ============================================================

def weighted_moving_average_forecast(
    values,
    window=30
):
    """
    Daha yeni maçlara daha yüksek ağırlık verir.
    """

    arr = np.asarray(values, dtype=float)

    if len(arr) == 0:
        return 0.0

    arr = arr[-window:]

    weights = np.arange(
        1,
        len(arr) + 1
    )

    return float(
        np.sum(arr * weights)
        / np.sum(weights)
    )


def ewma_forecast(
    values,
    span=3
):
    """
    Exponentially Weighted Moving Average (EWMA 3).
    """

    arr = np.asarray(values, dtype=float)

    if len(arr) == 0:
        return 0.0

    return float(
        pd.Series(arr)
        .ewm(
            span=min(span, len(arr)),
            adjust=False
        )
        .mean()
        .iloc[-1]
    )


def ar1_forecast(values):
    """
    Basit AR(1):
        y_t = c + phi*y_(t-1) + error
    """

    arr = np.asarray(
        values,
        dtype=float
    )

    if len(arr) < 3:
        return float(
            np.mean(arr)
            if len(arr)
            else 0.0
        )

    y = arr[1:]
    x = arr[:-1]

    X = np.column_stack([
        np.ones(len(x)),
        x
    ])

    try:

        beta = np.linalg.lstsq(
            X,
            y,
            rcond=None
        )[0]

        intercept = beta[0]
        phi = beta[1]

        forecast = (
            intercept
            + phi * arr[-1]
        )

        return float(
            max(0.01, forecast)
        )

    except Exception:

        return float(
            np.mean(arr[-7:])
        )


def poly_trend_forecast(
    values,
    degree=2
):
    """
    Polinom Trend Tahmini (2. Derece Polinom Uydurma)
    """

    arr = np.asarray(values, dtype=float)

    n = len(arr)

    if n == 0:
        return 0.0

    if n < degree + 1:
        return float(np.mean(arr))

    x = np.arange(n)

    try:
        coeffs = np.polyfit(x, arr, min(degree, n - 1))
        poly = np.poly1d(coeffs)
        pred = poly(n)
        return float(max(0.01, pred))
    except Exception:
        return float(np.mean(arr))


def linear_regression_forecast(values):
    """
    Zaman Serisi Üzerine Doğrudan Lineer Regresyon
    """

    arr = np.asarray(values, dtype=float)

    n = len(arr)

    if n == 0:
        return 0.0

    if n < 2:
        return float(arr[0])

    x = np.arange(n)

    try:
        slope, intercept = np.polyfit(x, arr, 1)
        pred = slope * n + intercept
        return float(max(0.01, pred))
    except Exception:
        return float(np.mean(arr))


# ============================================================
# 4. CLASSICAL MODEL BACKTEST
# ============================================================

def backtest_classical_model(
    series,
    method,
    min_train=30
):

    arr = np.asarray(
        series,
        dtype=float
    )

    if len(arr) <= min_train:
        return np.nan

    predictions = []
    actuals = []

    for t in range(
        min_train,
        len(arr)
    ):

        history = arr[:t]
        actual = arr[t]

        if method == "WMA":
            pred = weighted_moving_average_forecast(history, window=30)
        elif method == "EWMA(3)":
            pred = ewma_forecast(history, span=3)
        elif method == "AR(1)":
            pred = ar1_forecast(history)
        elif method == "Polinom Trend":
            pred = poly_trend_forecast(history, degree=2)
        elif method == "Lineer Regresyon":
            pred = linear_regression_forecast(history)
        elif method == "Ensemble (xRuns)":
            p1 = weighted_moving_average_forecast(history, window=30)
            p2 = ewma_forecast(history, span=3)
            p3 = ar1_forecast(history)
            p4 = poly_trend_forecast(history, degree=2)
            p5 = linear_regression_forecast(history)
            pred = float(np.mean([p1, p2, p3, p4, p5]))
        else:
            continue

        predictions.append(pred)
        actuals.append(actual)

    if not predictions:
        return np.nan

    return float(
        np.sqrt(
            mean_squared_error(
                actuals,
                predictions
            )
        )
    )


def classical_forecast_summary(
    series
):

    p_wma = weighted_moving_average_forecast(series, window=30)
    p_ewma = ewma_forecast(series, span=3)
    p_ar1 = ar1_forecast(series)
    p_poly = poly_trend_forecast(series, degree=2)
    p_lin = linear_regression_forecast(series)
    p_ens5 = float(np.mean([p_wma, p_ewma, p_ar1, p_poly, p_lin]))

    preds = {
        "WMA": p_wma,
        "EWMA(3)": p_ewma,
        "AR(1)": p_ar1,
        "Polinom Trend": p_poly,
        "Lineer Regresyon": p_lin,
        "Ensemble (xRuns)": p_ens5
    }

    methods = [
        "WMA",
        "EWMA(3)",
        "AR(1)",
        "Polinom Trend",
        "Lineer Regresyon",
        "Ensemble (xRuns)"
    ]

    rows = []

    for method in methods:
        forecast = preds[method]
        rmse = backtest_classical_model(series, method)

        rows.append({
            "Method": method,
            "Forecast": forecast,
            "RMSE": rmse
        })

    return pd.DataFrame(rows)


# ============================================================
# 5. DATASET CREATION
# ============================================================

def parse_series(
    text,
    expected_min=1,
    max_len=None
):

    values = [
        float(x.strip())
        for x in str(text).split(",")
        if x.strip()
    ]

    if len(values) < expected_min:

        raise ValueError(
            f"En az {expected_min} gözlem girilmeli."
        )

    if max_len is not None and len(values) > max_len:

        values = values[-max_len:]

    return values


def build_training_rows_from_100_game_logs(
    runs_100,
    window_size=30
):

    if len(runs_100) <= window_size:

        raise ValueError(
            f"Eğitim için en az "
            f"{window_size + 1} maç gerekiyor."
        )

    rows = []
    targets = []

    for t in range(
        window_size,
        len(runs_100)
    ):

        history = runs_100[
            t - window_size:t
        ]

        features = (
            MLFeatureExtractor
            .extract_time_series_features(
                history,
                prefix="runs"
            )
        )

        rows.append(features)
        targets.append(
            runs_100[t]
        )

    return (
        pd.DataFrame(rows),
        np.asarray(targets)
    )


def augment_training_data(
    X,
    team_stats,
    league_stats,
    opponent_pitching_context
):

    # Offensive form
    saber = (
        MLFeatureExtractor
        .extract_sabermetric_features(
            team_stats,
            league_stats
        )
    )

    # Opponent SP + Bullpen defense
    pitching = (
        MLFeatureExtractor
        .add_pitching_context(
            {},
            opponent_pitching_context
        )
    )

    saber.update(pitching)

    saber_df = pd.DataFrame(
        [saber] * len(X),
        index=X.index
    )

    return pd.concat(
        [
            X.reset_index(drop=True),
            saber_df.reset_index(drop=True)
        ],
        axis=1
    )


# ============================================================
# 6. TIME-SERIES VALIDATION
# ============================================================

def evaluate_time_series_model(
    model,
    X,
    y,
    n_splits=5
):

    if len(X) < 12:
        return None, None

    n_splits = min(
        n_splits,
        max(
            2,
            len(X) // 8
        )
    )

    tscv = TimeSeriesSplit(
        n_splits=n_splits
    )

    predictions = []
    actuals = []

    for train_idx, test_idx in tscv.split(X):

        fold_model = clone(model)

        fold_model.fit(
            X.iloc[train_idx],
            y[train_idx]
        )

        pred = fold_model.predict(
            X.iloc[test_idx]
        )

        predictions.extend(
            pred.tolist()
        )

        actuals.extend(
            y[test_idx].tolist()
        )

    actuals = np.asarray(
        actuals
    )

    predictions = np.asarray(
        predictions
    )

    return {
        "MAE": mean_absolute_error(
            actuals,
            predictions
        ),

        "RMSE": np.sqrt(
            mean_squared_error(
                actuals,
                predictions
            )
        ),

        "R2": (
            r2_score(
                actuals,
                predictions
            )
            if len(np.unique(actuals)) > 1
            else np.nan
        )
    }, predictions


def train_best_model(
    X,
    y,
    model_names
):

    results = []
    fitted = {}

    for name in model_names:

        model = model_factory(
            name
        )

        metrics, _ = (
            evaluate_time_series_model(
                model,
                X,
                y
            )
        )

        if metrics:

            results.append({
                "Model": name,
                **metrics
            })

            final_model = clone(
                model
            )

            final_model.fit(
                X,
                y
            )

            fitted[name] = final_model

    result_df = pd.DataFrame(
        results
    )

    if not result_df.empty:

        best_name = (
            result_df
            .sort_values("RMSE")
            .iloc[0]["Model"]
        )

    else:

        best_name = model_names[0]

    return (
        result_df,
        fitted,
        best_name
    )


# ============================================================
# 7. ENSEMBLE
# ============================================================

def inverse_rmse_weights(
    forecast_df
):

    df = forecast_df.copy()

    valid = df[
        df["RMSE"].notna()
        & (df["RMSE"] > 0)
    ].copy()

    if valid.empty:

        df["Weight"] = (
            1.0 / len(df)
        )

        return df

    valid["RawWeight"] = (
        1.0
        / valid["RMSE"]
    )

    total = valid[
        "RawWeight"
    ].sum()

    valid["Weight"] = (
        valid["RawWeight"]
        / total
    )

    df["Weight"] = 0.0

    for idx in valid.index:

        df.loc[
            idx,
            "Weight"
        ] = valid.loc[
            idx,
            "Weight"
        ]

    return df


def build_ensemble_forecast(
    classical_df,
    ml_forecast,
    ml_rmse
):

    rows = []

    # Classical models + 5'li Ensemble
    for _, row in classical_df.iterrows():

        rows.append({
            "Method": row["Method"],
            "Forecast": row["Forecast"],
            "RMSE": row["RMSE"]
        })

    # ML
    rows.append({
        "Method": "ML Skoru",
        "Forecast": ml_forecast,
        "RMSE": ml_rmse
    })

    ensemble_df = pd.DataFrame(
        rows
    )

    ensemble_df = (
        inverse_rmse_weights(
            ensemble_df
        )
    )

    final_forecast = float(
        np.sum(
            ensemble_df["Forecast"]
            * ensemble_df["Weight"]
        )
    )

    return (
        ensemble_df,
        final_forecast
    )


# ============================================================
# 8. STREAMLIT APP
# ============================================================

st.set_page_config(
    page_title="MLB Match Predictor",
    layout="wide",
    page_icon="⚾"
)

st.title(
    "⚾Maç Projeksiyonu"
)

st.caption(
    "Hücum Formu + Starting Pitcher Formu (xERA/xFIP) + "
    "Bullpen Formu (xERA/xFIP) + WMA + EWMA(3) + AR(1) + "
    "Polinom Trend + Lineer Regresyon + 5'li Ensemble + ML Skoru + Monte Carlo"
)


# ============================================================
# SIDEBAR
# ============================================================

st.sidebar.header(
    "⚙️Ayarlar"
)

models_list = [
    "Random Forest",
    "Gradient Boosting",
    "HistGradient Boosting",
    "Ridge Regression",
    "Poisson Regression"
]

if XGB_AVAILABLE:
    models_list.append(
        "XGBoost"
    )

model_choice = st.sidebar.selectbox(
    "Tercih Edilen Model",
    models_list
)

auto_select = st.sidebar.checkbox(
    "Walk-forward testinde en iyiyi otomatik seç",
    value=True
)

iters = st.sidebar.number_input(
    "Monte Carlo İterasyonu",
    1000,
    50000,
    10000,
    1000
)

park_factor = st.sidebar.number_input(
    "Park Factor (Stadyum Etkisi)",
    70.0,
    130.0,
    100.0,
    1.0
)


# ============================================================
# LEAGUE BASELINES
# ============================================================

league_stats = {

    "xwOBA": {
        "mean": 0.320,
        "std": 0.025
    },

    "wRC+": {
        "mean": 100.0,
        "std": 15.0
    },

    "OPS": {
        "mean": 0.720,
        "std": 0.080
    }
}


st.markdown("---")


# ============================================================
# MATCH INPUTS
# ============================================================

col_away, col_home = st.columns(2)


# ============================================================
# AWAY
# ============================================================

with col_away:

    st.header(
        "✈️Away"
    )

    away_runs_100 = st.text_area(
        "Son 100 Maç Skorları (Runs)",
        value="",
        placeholder="Örnek: 4,2,1,5,0,3,7,2,1,4,...",
        help="Away takımının maç skorlarını eskiden yeniye doğru virgülle ayırarak manuel gir. En az 31, en fazla 100 maç girilebilir.",
        height=100,
        key="away_runs_100_manual"
    )

    a_tab1, a_tab2, a_tab3 = st.tabs([
        "Son 30 Maç",
        "Son 7 Maç",
        "Son 5 (H2H)"
    ])

    with a_tab1:

        away_runs_30 = st.text_input(
            "Away Son 30",
            value=(
                "4,2,1,5,0,3,7,2,1,4,"
                "5,2,3,6,1,0,4,2,5,3,"
                "1,7,8,2,0,3,4,1,2,5"
            )
        )

    with a_tab2:

        away_runs_7 = st.text_input(
            "Away Son 7",
            value="3,4,1,2,5,0,4"
        )

    with a_tab3:

        away_runs_5 = st.text_input(
            "Away Son 5 (H2H)",
            value="2,4,3,1,5"
        )


    # ========================================================
    # AWAY OFFENSE
    # ========================================================

    st.markdown(
        "#### Batting Stats (Dep)"
    )

    df_away_saber = st.data_editor(

        pd.DataFrame(
            {
                "xwOBA": [
                    0.325,
                    0.330,
                    0.310
                ],

                "wRC+": [
                    105,
                    112,
                    95
                ],

                "OPS": [
                    0.735,
                    0.760,
                    0.700
                ]
            },

            index=[
                "season",
                "l30",
                "l7"
            ]
        ),

        use_container_width=True,

        key="saber_away"
    )


    # ========================================================
    # AWAY OPPONENT DEFENSE
    # ========================================================

    st.markdown(
        "#### Pitching Stats (Ev)"
    )

    st.markdown(
        "**Starting Pitcher**"
    )

    df_away_sp = st.data_editor(

        pd.DataFrame(
            {
                "xERA": [
                    3.80,
                    3.65,
                    3.90
                ],

                "xFIP": [
                    3.75,
                    3.60,
                    3.95
                ]
            },

            index=[
                "season",
                "l30",
                "l7"
            ]
        ),

        use_container_width=True,

        key="sp_away_opponent"
    )


    st.markdown(
        "**Bullpen**"
    )

    df_away_bp = st.data_editor(

        pd.DataFrame(
            {
                "xERA": [
                    4.10,
                    3.95,
                    4.25
                ],

                "xFIP": [
                    4.00,
                    3.85,
                    4.15
                ]
            },

            index=[
                "season",
                "l30",
                "l7"
            ]
        ),

        use_container_width=True,

        key="bp_away_opponent"
    )


# ============================================================
# HOME
# ============================================================

with col_home:

    st.header(
        "🏠Home"
    )

    home_runs_100 = st.text_area(
        "Son 100 Maç Skorları (Runs)",
        value="",
        placeholder="Örnek: 3,5,2,4,1,6,2,3,4,1,...",
        help="Home takımının maç skorlarını eskiden yeniye doğru virgülle ayırarak manuel gir. En az 31, en fazla 100 maç girilebilir.",
        height=100,
        key="home_runs_100_manual"
    )

    h_tab1, h_tab2, h_tab3 = st.tabs([
        "Son 30 Maç",
        "Son 7 Maç",
        "Son 5 (H2H)"
    ])

    with h_tab1:

        home_runs_30 = st.text_input(
            "Home Son 30",
            value=(
                "3,5,2,4,1,6,2,3,4,1,"
                "0,5,6,2,3,4,1,2,5,7,"
                "3,2,1,4,5,6,3,2,1,4"
            )
        )

    with h_tab2:

        home_runs_7 = st.text_input(
            "Home Son 7",
            value="5,6,3,2,1,4,3"
        )

    with h_tab3:

        home_runs_5 = st.text_input(
            "Home Son 5 (H2H)",
            value="4,1,2,6,3"
        )


    # ========================================================
    # HOME OFFENSE
    # ========================================================

    st.markdown(
        "#### Batting Stats (EV)"
    )

    df_home_saber = st.data_editor(

        pd.DataFrame(
            {
                "xwOBA": [
                    0.315,
                    0.305,
                    0.340
                ],

                "wRC+": [
                    98,
                    95,
                    115
                ],

                "OPS": [
                    0.710,
                    0.690,
                    0.770
                ]
            },

            index=[
                "season",
                "l30",
                "l7"
            ]
        ),

        use_container_width=True,

        key="saber_home"
    )


    # ========================================================
    # HOME OPPONENT DEFENSE
    # ========================================================

    st.markdown(
        "#### Pitching Stats (Dep)"
    )

    st.markdown(
        "**Starting Pitcher**"
    )

    df_home_sp = st.data_editor(

        pd.DataFrame(
            {
                "xERA": [
                    4.20,
                    4.05,
                    4.35
                ],

                "xFIP": [
                    4.05,
                    3.95,
                    4.40
                ]
            },

            index=[
                "season",
                "l30",
                "l7"
            ]
        ),

        use_container_width=True,

        key="sp_home_opponent"
    )


    st.markdown(
        "**Bullpen**"
    )

    df_home_bp = st.data_editor(

        pd.DataFrame(
            {
                "xERA": [
                    3.90,
                    3.75,
                    4.10
                ],

                "xFIP": [
                    3.85,
                    3.70,
                    4.05
                ]
            },

            index=[
                "season",
                "l30",
                "l7"
            ]
        ),

        use_container_width=True,

        key="bp_home_opponent"
    )


# ============================================================
# PREDICTION
# ============================================================

st.markdown("---")


if st.button(
    "🚀Maçı Simüle Et",
    use_container_width=True
):

    with st.spinner(
        "Forecasting modelleri, ML ve Monte Carlo çalıştırılıyor..."
    ):

        try:

            # =================================================
            # PARSE
            # =================================================

            a_100 = parse_series(
                away_runs_100,
                31,
                100
            )

            h_100 = parse_series(
                home_runs_100,
                31,
                100
            )


            # =================================================
            # ML TRAINING DATA
            # =================================================

            X_train_away, y_away = (
                build_training_rows_from_100_game_logs(
                    a_100,
                    30
                )
            )

            X_train_home, y_home = (
                build_training_rows_from_100_game_logs(
                    h_100,
                    30
                )
            )


            # =================================================
            # PITCHING CONTEXT
            # =================================================

            away_context = {

                "sp_xera_season":
                    df_away_sp.loc[
                        "season",
                        "xERA"
                    ],

                "sp_xera_l30":
                    df_away_sp.loc[
                        "l30",
                        "xERA"
                    ],

                "sp_xera_l7":
                    df_away_sp.loc[
                        "l7",
                        "xERA"
                    ],

                "sp_xfip_season":
                    df_away_sp.loc[
                        "season",
                        "xFIP"
                    ],

                "sp_xfip_l30":
                    df_away_sp.loc[
                        "l30",
                        "xFIP"
                    ],

                "sp_xfip_l7":
                    df_away_sp.loc[
                        "l7",
                        "xFIP"
                    ],

                "bp_xera_season":
                    df_away_bp.loc[
                        "season",
                        "xERA"
                    ],

                "bp_xera_l30":
                    df_away_bp.loc[
                        "l30",
                        "xERA"
                    ],

                "bp_xera_l7":
                    df_away_bp.loc[
                        "l7",
                        "xERA"
                    ],

                "bp_xfip_season":
                    df_away_bp.loc[
                        "season",
                        "xFIP"
                    ],

                "bp_xfip_l30":
                    df_away_bp.loc[
                        "l30",
                        "xFIP"
                    ],

                "bp_xfip_l7":
                    df_away_bp.loc[
                        "l7",
                        "xFIP"
                    ],

                "home_advantage": 0,

                "park_factor":
                    park_factor
            }


            home_context = {

                "sp_xera_season":
                    df_home_sp.loc[
                        "season",
                        "xERA"
                    ],

                "sp_xera_l30":
                    df_home_sp.loc[
                        "l30",
                        "xERA"
                    ],

                "sp_xera_l7":
                    df_home_sp.loc[
                        "l7",
                        "xERA"
                    ],

                "sp_xfip_season":
                    df_home_sp.loc[
                        "season",
                        "xFIP"
                    ],

                "sp_xfip_l30":
                    df_home_sp.loc[
                        "l30",
                        "xFIP"
                    ],

                "sp_xfip_l7":
                    df_home_sp.loc[
                        "l7",
                        "xFIP"
                    ],

                "bp_xera_season":
                    df_home_bp.loc[
                        "season",
                        "xERA"
                    ],

                "bp_xera_l30":
                    df_home_bp.loc[
                        "l30",
                        "xERA"
                    ],

                "bp_xera_l7":
                    df_home_bp.loc[
                        "l7",
                        "xERA"
                    ],

                "bp_xfip_season":
                    df_home_bp.loc[
                        "season",
                        "xFIP"
                    ],

                "bp_xfip_l30":
                    df_home_bp.loc[
                        "l30",
                        "xFIP"
                    ],

                "bp_xfip_l7":
                    df_home_bp.loc[
                        "l7",
                        "xFIP"
                    ],

                "home_advantage": 1,

                "park_factor":
                    park_factor
            }


            # =================================================
            # AUGMENT ML DATA
            # =================================================

            X_train_away = augment_training_data(
                X_train_away,
                df_away_saber.to_dict(),
                league_stats,
                away_context
            )

            X_train_home = augment_training_data(
                X_train_home,
                df_home_saber.to_dict(),
                league_stats,
                home_context
            )


            # =================================================
            # PREDICTION VECTOR
            # =================================================

            def build_pred_vector(
                runs_30_txt,
                runs_7_txt,
                runs_5_txt,
                saber_dict,
                context,
                X_train_ref
            ):

                r30 = parse_series(
                    runs_30_txt
                )

                r7 = parse_series(
                    runs_7_txt
                )

                r5 = parse_series(
                    runs_5_txt
                )

                pred_feats = (
                    MLFeatureExtractor
                    .extract_time_series_features(
                        r30,
                        prefix="runs"
                    )
                )

                pred_feats.update({

                    "runs_l7_mean":
                        float(
                            np.mean(r7)
                        ),

                    "runs_l5_h2h_mean":
                        float(
                            np.mean(r5)
                        ),

                    "runs_l5_h2h_max":
                        float(
                            np.max(r5)
                        )
                })

                pred_feats.update(
                    MLFeatureExtractor
                    .extract_sabermetric_features(
                        saber_dict,
                        league_stats
                    )
                )

                pred_feats.update(
                    MLFeatureExtractor
                    .add_pitching_context(
                        {},
                        context
                    )
                )

                return (
                    pd.DataFrame(
                        [pred_feats]
                    )
                    .reindex(
                        columns=X_train_ref.columns,
                        fill_value=0.0
                    )
                )


            X_pred_away = build_pred_vector(
                away_runs_30,
                away_runs_7,
                away_runs_5,
                df_away_saber.to_dict(),
                away_context,
                X_train_away
            )

            X_pred_home = build_pred_vector(
                home_runs_30,
                home_runs_7,
                home_runs_5,
                df_home_saber.to_dict(),
                home_context,
                X_train_home
            )


            # =================================================
            # ML MODELS
            # =================================================

            res_away, models_away, best_away = (
                train_best_model(
                    X_train_away,
                    y_away,
                    models_list
                )
            )

            res_home, models_home, best_home = (
                train_best_model(
                    X_train_home,
                    y_home,
                    models_list
                )
            )


            sel_away = (
                best_away
                if auto_select
                else model_choice
            )

            sel_home = (
                best_home
                if auto_select
                else model_choice
            )


            model_a = models_away.get(
                sel_away,
                models_away[best_away]
            )

            model_h = models_home.get(
                sel_home,
                models_home[best_home]
            )


            # =================================================
            # RAW ML FORECAST
            # =================================================

            ml_lambda_away = max(
                0.01,
                float(
                    model_a.predict(
                        X_pred_away
                    )[0]
                )
            )

            ml_lambda_home = max(
                0.01,
                float(
                    model_h.predict(
                        X_pred_home
                    )[0]
                )
            )


            # =================================================
            # ML RMSE
            # =================================================

            ml_rmse_away = float(
                res_away.loc[
                    res_away["Model"] == sel_away,
                    "RMSE"
                ].iloc[0]
            )

            ml_rmse_home = float(
                res_home.loc[
                    res_home["Model"] == sel_home,
                    "RMSE"
                ].iloc[0]
            )


            # =================================================
            # CLASSICAL FORECASTING
            # =================================================

            classical_away = (
                classical_forecast_summary(
                    a_100
                )
            )

            classical_home = (
                classical_forecast_summary(
                    h_100
                )
            )


            # =================================================
            # ENSEMBLE
            # =================================================

            ensemble_away, lambda_away = (
                build_ensemble_forecast(
                    classical_away,
                    ml_lambda_away,
                    ml_rmse_away
                )
            )

            ensemble_home, lambda_home = (
                build_ensemble_forecast(
                    classical_home,
                    ml_lambda_home,
                    ml_rmse_home
                )
            )


            lambda_away = max(
                0.01,
                lambda_away
            )

            lambda_home = max(
                0.01,
                lambda_home
            )


            # =================================================
            # MONTE CARLO
            # =================================================

            sim_away = np.random.poisson(
                lambda_away,
                iters
            )

            sim_home = np.random.poisson(
                lambda_home,
                iters
            )

            sim_total = (
                sim_away
                + sim_home
            )


            # =================================================
            # PROBABILITIES
            # =================================================

            away_win_prob = (
                np.mean(
                    sim_away > sim_home
                ) * 100
            )

            home_win_prob = (
                np.mean(
                    sim_home > sim_away
                ) * 100
            )

            tie_prob = (
                np.mean(
                    sim_away == sim_home
                ) * 100
            )

            over_8_5 = (
                np.mean(
                    sim_total > 8.5
                ) * 100
            )


            # =================================================
            # OUTPUT
            # =================================================

            st.success(
                "Forecasting, Walk-Forward Validation, "
                "Ensemble ve Monte Carlo tamamlandı!"
            )


            # =================================================
            # MAIN METRICS
            # =================================================

            c1, c2, c3, c4 = st.columns(4)

            c1.metric(
                "Away - Beklenen Skor",
                f"{lambda_away:.2f}"
            )

            c2.metric(
                "Home - Beklenen Skor",
                f"{lambda_home:.2f}"
            )

            c3.metric(
                "Toplam Beklenen Runs",
                f"{lambda_away + lambda_home:.2f}"
            )

            c4.metric(
                "📈 8.5 Üst Olasılığı",
                f"%{over_8_5:.1f}"
            )


            # =================================================
            # FORECAST BREAKDOWN
            # =================================================

            st.markdown("---")

            st.subheader(
                "Tahmin Motorlarının Ayrışması"
            )

            fc1, fc2 = st.columns(2)


            with fc1:

                st.markdown(
                    "### ✈️Away"
                )

                display_away = (
                    ensemble_away[
                        [
                            "Method",
                            "Forecast",
                            "RMSE",
                            "Weight"
                        ]
                    ].copy()
                )

                st.dataframe(
                    display_away.style.format({
                        "Forecast": "{:.3f}",
                        "RMSE": "{:.3f}",
                        "Weight": "{:.2%}"
                    }),
                    hide_index=True,
                    use_container_width=True
                )


            with fc2:

                st.markdown(
                    "### 🏠Home"
                )

                display_home = (
                    ensemble_home[
                        [
                            "Method",
                            "Forecast",
                            "RMSE",
                            "Weight"
                        ]
                    ].copy()
                )

                st.dataframe(
                    display_home.style.format({
                        "Forecast": "{:.3f}",
                        "RMSE": "{:.3f}",
                        "Weight": "{:.2%}"
                    }),
                    hide_index=True,
                    use_container_width=True
                )


            # =================================================
            # MODEL EXPLANATION
            # =================================================

            st.markdown("---")

            st.subheader(
                "🧠Final Ensemble Yapısı"
            )

            st.write(
                f"""
                **Away:**
                WMA + EWMA(3) + AR(1) + Polinom Trend + Lineer Regresyon + 5'li Ensemble + ML Skoru → **{lambda_away:.2f}**

                **Home:**
                WMA + EWMA(3) + AR(1) + Polinom Trend + Lineer Regresyon + 5'li Ensemble + ML Skoru → **{lambda_home:.2f}**

                Ensemble ağırlıkları geçmiş backtest RMSE
                değerlerinin tersine göre belirlenmiştir.
                Daha düşük RMSE → daha yüksek ağırlık.
                """
            )


            # =================================================
            # WIN PROBABILITIES
            # =================================================

            st.markdown(
                "### 🏆Kazanma Olasılıkları"
            )

            st.progress(
                away_win_prob / 100
            )

            st.write(
                f"""
                **Deplasman:** %{away_win_prob:.1f}
                **Beraberlik/Uzatma:** %{tie_prob:.1f}
                **Ev Sahibi:** %{home_win_prob:.1f}
                """
            )


            # =================================================
            # MONTE CARLO DISTRIBUTIONS
            # =================================================

            col_chart1, col_chart2 = st.columns(2)


            with col_chart1:

                df_dist = pd.DataFrame({

                    "Score":
                        np.concatenate([
                            sim_away,
                            sim_home
                        ]),

                    "Team":
                        (
                            ["Away"] * iters
                            + ["Home"] * iters
                        )
                })

                fig1 = px.histogram(
                    df_dist,
                    x="Score",
                    color="Team",
                    barmode="group",
                    title=(
                        "Takım Skor Dağılımı "
                        "(Monte Carlo)"
                    ),
                    histnorm="percent"
                )

                st.plotly_chart(
                    fig1,
                    use_container_width=True
                )


            with col_chart2:

                fig2 = px.histogram(
                    sim_total,
                    title=(
                        "Toplam Maç Skoru "
                        "Dağılımı"
                    ),
                    labels={
                        "value":
                            "Toplam Sayı"
                    },
                    histnorm="percent"
                )

                fig2.add_vline(
                    x=8.5,
                    line_dash="dash",
                    annotation_text="8.5 Line"
                )

                st.plotly_chart(
                    fig2,
                    use_container_width=True
                )


            # =================================================
            # VALIDATION
            # =================================================

            st.markdown("---")

            st.subheader(
                "📊ML Model Doğrulama Performansı"
            )

            v1, v2 = st.columns(2)


            with v1:

                st.write(
                    f"**Away Seçilen Model:** "
                    f"{sel_away}"
                )

                st.dataframe(
                    res_away.style.format({
                        "MAE": "{:.3f}",
                        "RMSE": "{:.3f}",
                        "R2": "{:.3f}"
                    }),
                    hide_index=True,
                    use_container_width=True
                )


            with v2:

                st.write(
                    f"**Home Seçilen Model:** "
                    f"{sel_home}"
                )

                st.dataframe(
                    res_home.style.format({
                        "MAE": "{:.3f}",
                        "RMSE": "{:.3f}",
                        "R2": "{:.3f}"
                    }),
                    hide_index=True,
                    use_container_width=True
                )


            # =================================================
            # CLASSICAL MODEL VALIDATION
            # =================================================

            st.markdown("---")

            st.subheader(
                "📈Klasik Forecasting & Ensemble Backtest"
            )

            classical_display_away = (
                classical_away.copy()
            )

            classical_display_home = (
                classical_home.copy()
            )


            c1, c2 = st.columns(2)

            with c1:

                st.write(
                    "**Away Classical Forecasting**"
                )

                st.dataframe(
                    classical_display_away.style.format({
                        "Forecast": "{:.3f}",
                        "RMSE": "{:.3f}"
                    }),
                    hide_index=True,
                    use_container_width=True
                )


            with c2:

                st.write(
                    "**Home Classical Forecasting**"
                )

                st.dataframe(
                    classical_display_home.style.format({
                        "Forecast": "{:.3f}",
                        "RMSE": "{:.3f}"
                    }),
                    hide_index=True,
                    use_container_width=True
                )


            # =================================================
            # SHAP
            # =================================================

            st.markdown("---")

            st.subheader(
                "🔎Feature Importance & "
                "Model Karar Mekanizması"
            )


            if (
                SHAP_AVAILABLE
                and hasattr(
                    model_a,
                    "feature_importances_"
                )
            ):

                st.write(
                    "Deplasman modeli için SHAP"
                )

                try:

                    explainer = (
                        shap.TreeExplainer(
                            model_a
                        )
                    )

                    shap_values = (
                        explainer.shap_values(
                            X_train_away
                        )
                    )

                    fig_shap, ax = plt.subplots(
                        figsize=(8, 4)
                    )

                    shap.summary_plot(
                        shap_values,
                        X_train_away,
                        max_display=12,
                        show=False
                    )

                    st.pyplot(
                        fig_shap
                    )

                except Exception:

                    st.warning(
                        "Ağaç tabanlı SHAP çizilemedi."
                    )


            elif hasattr(
                model_a,
                "feature_importances_"
            ):

                fi = pd.DataFrame({

                    "Feature":
                        X_train_away.columns,

                    "Importance":
                        model_a.feature_importances_

                }).sort_values(
                    "Importance",
                    ascending=False
                )

                fig_fi = px.bar(
                    fi.head(12)
                    .sort_values(
                        "Importance"
                    ),

                    x="Importance",
                    y="Feature",
                    orientation="h",

                    title=(
                        "Top 12 Etkili Faktör "
                        "(Deplasman)"
                    )
                )

                st.plotly_chart(
                    fig_fi,
                    use_container_width=True
                )


        except Exception as e:

            st.error(
                f"Tahmin işlemi sırasında hata: {e}"
            )

            st.exception(e)

In [ ]:
#Postseason Maç Analizi (ML)
%%writefile sabermetrics_gui.py
import streamlit as st
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.base import clone
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor
)
from sklearn.linear_model import Ridge, PoissonRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from pathlib import Path

# CSV aynı klasördeyse doğrudan buradan okunur.
# Dosya adı: MaçML.csv
POSTSEASON_CSV = Path(__file__).with_name("MaçML.csv")
POSTSEASON_WEIGHT = 0.06  # Postseason geçmişinin ML çıktısına maksimum yaklaşık ±%6 etkisi


# Optional XGBoost ve SHAP
try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False

try:
    import shap
    import matplotlib.pyplot as plt
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False


# ============================================================
# 1. FEATURE ENGINEERING
# ============================================================

class MLFeatureExtractor:
    """
    Maç tahminine uyarlanmış özellik çıkarıcı.

    Hücum:
        xwOBA / wRC+ / OPS
        Season / L30 / L7

    Savunma:
        Starting Pitcher:
            ERA / FIP
            Season / L30 / L7

        Bullpen:
            ERA / FIP
            Season / L30 / L7
    """

    @staticmethod
    def extract_time_series_features(scores, prefix="runs"):
        arr = np.asarray(scores, dtype=float)

        if len(arr) == 0:
            arr = np.array([0.0])

        n = len(arr)
        x = np.arange(n)

        features = {
            f"{prefix}_mean": float(np.mean(arr)),
            f"{prefix}_std": float(np.std(arr, ddof=0)),
            f"{prefix}_max": float(np.max(arr)),
            f"{prefix}_min": float(np.min(arr)),
            f"{prefix}_median": float(np.median(arr)),

            # EWMA
            f"{prefix}_ewma": float(
                pd.Series(arr)
                .ewm(span=min(7, n), adjust=False)
                .mean()
                .iloc[-1]
            ),
        }

        # Trend
        if n > 1:
            slope, _ = np.polyfit(x, arr, 1)

            features[f"{prefix}_slope"] = float(slope)
            features[f"{prefix}_recent_delta"] = float(
                arr[-1] - arr[0]
            )
        else:
            features[f"{prefix}_slope"] = 0.0
            features[f"{prefix}_recent_delta"] = 0.0

        # Rolling windows
        for window in [3, 7, 15, 30]:

            if n >= window:
                recent = arr[-window:]
            else:
                recent = arr

            features[f"{prefix}_l{window}_mean"] = float(
                np.mean(recent)
            )

            features[f"{prefix}_l{window}_std"] = float(
                np.std(recent, ddof=0)
            )

            features[f"{prefix}_l{window}_ewma"] = float(
                pd.Series(recent)
                .ewm(
                    span=min(7, len(recent)),
                    adjust=False
                )
                .mean()
                .iloc[-1]
            )

        features[f"{prefix}_last"] = float(arr[-1])

        return features

    # --------------------------------------------------------
    # OFFENSE
    # --------------------------------------------------------

    @staticmethod
    def extract_sabermetric_features(team_stats, league_stats):

        feats = {}

        metrics = [
            "xwOBA",
            "wRC+",
            "OPS"
        ]

        for metric in metrics:

            mean_val = float(
                league_stats[metric]["mean"]
            )

            std_val = float(
                league_stats[metric]["std"]
            )

            season = float(
                team_stats[metric]["season"]
            )

            l30 = float(
                team_stats[metric]["l30"]
            )

            l7 = float(
                team_stats[metric]["l7"]
            )

            for label, value in [
                ("season", season),
                ("l30", l30),
                ("l7", l7)
            ]:

                feats[
                    f"{metric}_z_{label}"
                ] = (
                    (value - mean_val) / std_val
                    if std_val > 0
                    else 0.0
                )

            feats[
                f"{metric}_delta_l7_season"
            ] = l7 - season

            feats[
                f"{metric}_delta_l30_season"
            ] = l30 - season

        return feats

    # --------------------------------------------------------
    # PITCHING
    # --------------------------------------------------------

    @staticmethod
    def extract_pitching_features(
        pitching_stats,
        prefix
    ):
        """
        prefix:
            sp
            bp

        Metrics:
            ERA
            FIP

        Periods:
            season
            l30
            l7
        """

        feats = {}

        metrics = [
            "ERA",
            "FIP"
        ]

        for metric in metrics:

            season = float(
                pitching_stats[metric]["season"]
            )

            l30 = float(
                pitching_stats[metric]["l30"]
            )

            l7 = float(
                pitching_stats[metric]["l7"]
            )

            for label, value in [
                ("season", season),
                ("l30", l30),
                ("l7", l7)
            ]:

                feats[
                    f"{prefix}_{metric.lower()}_{label}"
                ] = value

            feats[
                f"{prefix}_{metric.lower()}_delta_l7_season"
            ] = l7 - season

            feats[
                f"{prefix}_{metric.lower()}_delta_l30_season"
            ] = l30 - season

        return feats

    @staticmethod
    def add_pitching_context(
        features,
        context
    ):

        features.update({

            "opp_sp_era_season":
                float(context.get(
                    "sp_era_season", 4.0
                )),

            "opp_sp_era_l30":
                float(context.get(
                    "sp_era_l30", 4.0
                )),

            "opp_sp_era_l7":
                float(context.get(
                    "sp_era_l7", 4.0
                )),

            "opp_sp_fip_season":
                float(context.get(
                    "sp_fip_season", 4.0
                )),

            "opp_sp_fip_l30":
                float(context.get(
                    "sp_fip_l30", 4.0
                )),

            "opp_sp_fip_l7":
                float(context.get(
                    "sp_fip_l7", 4.0
                )),

            "opp_bp_era_season":
                float(context.get(
                    "bp_era_season", 4.2
                )),

            "opp_bp_era_l30":
                float(context.get(
                    "bp_era_l30", 4.2
                )),

            "opp_bp_era_l7":
                float(context.get(
                    "bp_era_l7", 4.2
                )),

            "opp_bp_fip_season":
                float(context.get(
                    "bp_fip_season", 4.2
                )),

            "opp_bp_fip_l30":
                float(context.get(
                    "bp_fip_l30", 4.2
                )),

            "opp_bp_fip_l7":
                float(context.get(
                    "bp_fip_l7", 4.2
                )),

            "home_advantage":
                float(context.get(
                    "home_advantage", 1
                )),

            "park_factor":
                float(context.get(
                    "park_factor", 100
                )),
        })

        return features


# ============================================================
# 2. POSTSEASON DATABASE
# ============================================================

@st.cache_data
def load_postseason_database(csv_path):
    """
    MaçML.csv header'sız 9 kolonlu bir veri tabanıdır:
      0 id
      1 team
      2 year
      3 postseason (0/1)
      4 postseason_games
      5 postseason_hits
      6 postseason_era
      7 postseason_fip
      8 champion (0/1)

    Ondalık sayılar CSV'de virgülle tutulduğu için normalize edilir.
    """
    if not Path(csv_path).exists():
        return None

    raw = pd.read_csv(
        csv_path,
        header=None,
        encoding="utf-8-sig",
        sep=None,
        engine="python"
    )

    if raw.shape[1] < 9:
        raise ValueError(
            f"MaçML.csv en az 9 kolon içermeli; bulunan: {raw.shape[1]}"
        )

    raw = raw.iloc[:, :9].copy()
    raw.columns = [
        "id", "team", "year", "postseason",
        "postseason_games", "postseason_hits",
        "postseason_era", "postseason_fip", "champion"
    ]

    for col in [
        "year", "postseason", "postseason_games",
        "postseason_hits", "postseason_era",
        "postseason_fip", "champion"
    ]:
        raw[col] = pd.to_numeric(
            raw[col].astype(str).str.replace(",", ".", regex=False),
            errors="coerce"
        )

    raw["team"] = raw["team"].astype(str).str.strip()
    raw["year"] = raw["year"].astype(int)

    return raw


def _minmax_series(s, inverse=False):
    s = pd.to_numeric(s, errors="coerce")
    if inverse:
        s = -s
    lo, hi = s.min(), s.max()
    if pd.isna(lo) or pd.isna(hi) or hi == lo:
        return pd.Series(50.0, index=s.index)
    return 100 * (s - lo) / (hi - lo)


def build_postseason_profiles(db):
    """
    Her takım için 2000+ postseason geçmişini tek bir 0-100 indeksine çevirir.

    Ağırlıklar:
      %25 postseason'a çıkma oranı
      %15 postseason oyun hacmi
      %10 postseason hit hacmi
      %20 postseason ERA (düşük daha iyi)
      %20 postseason FIP (düşük daha iyi)
      %10 şampiyonluk oranı

    Sezon ağırlığı: yeni sezonlara hafif daha yüksek ağırlık.
    """
    if db is None or db.empty:
        return pd.DataFrame()

    max_year = int(db["year"].max())
    d = db.copy()
    d["recency_w"] = np.exp(-0.06 * (max_year - d["year"]))

    rows = []
    for team, g in d.groupby("team"):
        postseason = g["postseason"].fillna(0)
        w = g["recency_w"]

        # Postseason performans metrikleri sadece postseason yapılan yıllardan hesaplanır.
        pg = g[g["postseason"] == 1].copy()
        pw = pg["recency_w"] if not pg.empty else pd.Series(dtype=float)

        def weighted_mean(col):
            if pg.empty:
                return np.nan
            vals = pd.to_numeric(pg[col], errors="coerce")
            mask = vals.notna() & pw.notna()
            if not mask.any():
                return np.nan
            return float(np.average(vals[mask], weights=pw[mask]))

        rows.append({
            "team": team,
            "postseason_rate": float(np.average(postseason, weights=w)),
            "postseason_games": weighted_mean("postseason_games"),
            "postseason_hits": weighted_mean("postseason_hits"),
            "postseason_era": weighted_mean("postseason_era"),
            "postseason_fip": weighted_mean("postseason_fip"),
            "championship_rate": float(np.average(
                g["champion"].fillna(0), weights=w
            )),
            "postseason_seasons": int(postseason.sum()),
            "titles": int(g["champion"].fillna(0).sum())
        })

    p = pd.DataFrame(rows)

    # Eksik postseason metriği varsa lig/ekip ortalamasıyla doldur.
    for col in [
        "postseason_games", "postseason_hits",
        "postseason_era", "postseason_fip"
    ]:
        p[col] = p[col].fillna(p[col].median())

    p["idx_appearance"] = _minmax_series(p["postseason_rate"])
    p["idx_games"] = _minmax_series(p["postseason_games"])
    p["idx_hits"] = _minmax_series(p["postseason_hits"])
    p["idx_era"] = _minmax_series(p["postseason_era"], inverse=True)
    p["idx_fip"] = _minmax_series(p["postseason_fip"], inverse=True)
    p["idx_title"] = _minmax_series(p["championship_rate"])

    p["postseason_index"] = (
        0.25 * p["idx_appearance"]
        + 0.15 * p["idx_games"]
        + 0.10 * p["idx_hits"]
        + 0.20 * p["idx_era"]
        + 0.20 * p["idx_fip"]
        + 0.10 * p["idx_title"]
    )

    return p.sort_values("postseason_index", ascending=False).reset_index(drop=True)


def get_postseason_profile(profiles, team):
    if profiles is None or profiles.empty:
        return None
    match = profiles[
        profiles["team"].str.casefold() == str(team).strip().casefold()
    ]
    if match.empty:
        return None
    return match.iloc[0].to_dict()


def add_postseason_ml_features(X, team_profile, opponent_profile):
    """
    Postseason geçmişini ML feature setine ekler.
    Böylece SHAP / feature importance ekranında görünür.
    """
    X = X.copy()

    tp = team_profile or {}
    op = opponent_profile or {}

    team_idx = float(tp.get("postseason_index", 50.0))
    opp_idx = float(op.get("postseason_index", 50.0))

    feature_values = {
        "postseason_index": team_idx,
        "opp_postseason_index": opp_idx,
        "postseason_matchup_edge": team_idx - opp_idx,
        "postseason_rate": float(tp.get("postseason_rate", 0.0)),
        "postseason_games": float(tp.get("postseason_games", 0.0)),
        "postseason_hits": float(tp.get("postseason_hits", 0.0)),
        "postseason_era": float(tp.get("postseason_era", 4.5)),
        "postseason_fip": float(tp.get("postseason_fip", 4.5)),
        "championship_rate": float(tp.get("championship_rate", 0.0)),
    }

    for key, value in feature_values.items():
        X[key] = value

    return X


def apply_postseason_weight(raw_lambda, team_profile, opponent_profile):
    """
    ML forecastu doğrudan ezmek yerine sınırlı bir matchup adjustment uygular.
    Edge +100 / -100 olsa bile etki ±%6'yı aşmaz.
    """
    if not team_profile or not opponent_profile:
        return float(raw_lambda), 1.0

    team_idx = float(team_profile.get("postseason_index", 50.0))
    opp_idx = float(opponent_profile.get("postseason_index", 50.0))

    edge = np.clip((team_idx - opp_idx) / 50.0, -1.0, 1.0)
    multiplier = 1.0 + POSTSEASON_WEIGHT * edge

    return float(max(0.01, raw_lambda * multiplier)), float(multiplier)


# ============================================================
# 3. MODEL FACTORY
# ============================================================

def model_factory(model_name):

    if model_name == "Random Forest":

        return RandomForestRegressor(
            n_estimators=300,
            max_depth=6,
            min_samples_leaf=4,
            random_state=42
        )

    if model_name == "Gradient Boosting":

        return GradientBoostingRegressor(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            loss="huber",
            random_state=42
        )

    if model_name == "HistGradient Boosting":

        return HistGradientBoostingRegressor(
            max_iter=250,
            learning_rate=0.05,
            l2_regularization=1.5,
            random_state=42
        )

    if model_name == "XGBoost" and XGB_AVAILABLE:

        return XGBRegressor(
            n_estimators=250,
            max_depth=3,
            learning_rate=0.05,
            objective="reg:squarederror",
            random_state=42
        )

    if model_name == "Poisson Regression":

        return Pipeline([
            (
                "scaler",
                StandardScaler()
            ),
            (
                "poisson",
                PoissonRegressor(
                    alpha=0.8,
                    max_iter=1000
                )
            )
        ])

    return Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "ridge",
            Ridge(alpha=5.0)
        )
    ])


# ============================================================
# 3. CLASSICAL FORECASTING MODELS
# ============================================================

def weighted_moving_average_forecast(
    values,
    window=30
):
    """
    Daha yeni maçlara daha yüksek ağırlık verir.
    """

    arr = np.asarray(values, dtype=float)

    if len(arr) == 0:
        return 0.0

    arr = arr[-window:]

    weights = np.arange(
        1,
        len(arr) + 1
    )

    return float(
        np.sum(arr * weights)
        / np.sum(weights)
    )


def ewma_forecast(
    values,
    span=7
):
    """
    Exponentially Weighted Moving Average.
    """

    arr = np.asarray(values, dtype=float)

    if len(arr) == 0:
        return 0.0

    return float(
        pd.Series(arr)
        .ewm(
            span=min(span, len(arr)),
            adjust=False
        )
        .mean()
        .iloc[-1]
    )


def linear_regression_forecast(values):
    """Zaman indeksine göre basit lineer trend ile bir sonraki gözlemi tahmin eder."""

    arr = np.asarray(values, dtype=float)

    if len(arr) == 0:
        return 0.0

    if len(arr) < 2:
        return float(arr[-1])

    x = np.arange(len(arr), dtype=float)
    slope, intercept = np.polyfit(x, arr, 1)

    forecast = intercept + slope * len(arr)

    return float(max(0.01, forecast))


def polynomial_trend_forecast(values, degree=2):
    """Zaman indeksine göre ikinci dereceden polinom trend ile bir sonraki gözlemi tahmin eder."""

    arr = np.asarray(values, dtype=float)

    if len(arr) == 0:
        return 0.0

    if len(arr) < 3:
        return linear_regression_forecast(arr)

    degree = min(degree, len(arr) - 1)
    x = np.arange(len(arr), dtype=float)

    try:
        coeffs = np.polyfit(x, arr, degree)
        forecast = np.polyval(coeffs, len(arr))
        return float(max(0.01, forecast))
    except Exception:
        return linear_regression_forecast(arr)


def ar1_forecast(values):
    """
    Basit AR(1):

        y_t = c + phi*y_(t-1) + error

    OLS ile phi ve intercept tahmin edilir.
    """

    arr = np.asarray(
        values,
        dtype=float
    )

    if len(arr) < 3:
        return float(
            np.mean(arr)
            if len(arr)
            else 0.0
        )

    y = arr[1:]
    x = arr[:-1]

    X = np.column_stack([
        np.ones(len(x)),
        x
    ])

    try:

        beta = np.linalg.lstsq(
            X,
            y,
            rcond=None
        )[0]

        intercept = beta[0]
        phi = beta[1]

        forecast = (
            intercept
            + phi * arr[-1]
        )

        return float(
            max(0.01, forecast)
        )

    except Exception:

        return float(
            np.mean(arr[-7:])
        )


# ============================================================
# 4. CLASSICAL MODEL BACKTEST
# ============================================================

def backtest_classical_model(
    series,
    method,
    min_train=30
):

    arr = np.asarray(
        series,
        dtype=float
    )

    if len(arr) <= min_train:
        return np.nan

    predictions = []
    actuals = []

    for t in range(
        min_train,
        len(arr)
    ):

        history = arr[:t]
        actual = arr[t]

        if method == "WMA":

            pred = weighted_moving_average_forecast(
                history,
                window=30
            )

        elif method == "EWMA(3)":

            pred = ewma_forecast(
                history,
                span=3
            )

        elif method == "Lineer Regresyon":

            pred = linear_regression_forecast(
                history
            )

        elif method == "Polinom Trend":

            pred = polynomial_trend_forecast(
                history,
                degree=2
            )

        elif method == "AR(1)":

            pred = ar1_forecast(
                history
            )

        else:

            continue

        predictions.append(pred)
        actuals.append(actual)

    if not predictions:
        return np.nan

    return float(
        np.sqrt(
            mean_squared_error(
                actuals,
                predictions
            )
        )
    )


def classical_forecast_summary(
    series
):

    methods = [
        "WMA",
        "EWMA(3)",
        "Lineer Regresyon",
        "Polinom Trend",
        "AR(1)"
    ]

    rows = []

    for method in methods:

        if method == "WMA":

            forecast = weighted_moving_average_forecast(
                series
            )

        elif method == "EWMA(3)":

            forecast = ewma_forecast(
                series,
                span=3
            )

        elif method == "Lineer Regresyon":

            forecast = linear_regression_forecast(
                series
            )

        elif method == "Polinom Trend":

            forecast = polynomial_trend_forecast(
                series,
                degree=2
            )

        else:

            forecast = ar1_forecast(
                series
            )

        rmse = backtest_classical_model(
            series,
            method
        )

        rows.append({
            "Method": method,
            "Forecast": forecast,
            "RMSE": rmse
        })

    return pd.DataFrame(rows)


# ============================================================
# 5. DATASET CREATION
# ============================================================

def parse_series(
    text,
    expected_min=1,
    max_len=None
):

    values = [
        float(x.strip())
        for x in str(text).split(",")
        if x.strip()
    ]

    if len(values) < expected_min:

        raise ValueError(
            f"En az {expected_min} gözlem girilmeli."
        )

    if max_len is not None and len(values) > max_len:

        values = values[-max_len:]

    return values


def build_training_rows_from_100_game_logs(
    runs_100,
    window_size=30
):

    if len(runs_100) <= window_size:

        raise ValueError(
            f"Eğitim için en az "
            f"{window_size + 1} maç gerekiyor."
        )

    rows = []
    targets = []

    for t in range(
        window_size,
        len(runs_100)
    ):

        history = runs_100[
            t - window_size:t
        ]

        features = (
            MLFeatureExtractor
            .extract_time_series_features(
                history,
                prefix="runs"
            )
        )

        rows.append(features)
        targets.append(
            runs_100[t]
        )

    return (
        pd.DataFrame(rows),
        np.asarray(targets)
    )


def augment_training_data(
    X,
    team_stats,
    league_stats,
    opponent_pitching_context
):

    # Offensive form
    saber = (
        MLFeatureExtractor
        .extract_sabermetric_features(
            team_stats,
            league_stats
        )
    )

    # Opponent SP + Bullpen defense
    pitching = (
        MLFeatureExtractor
        .add_pitching_context(
            {},
            opponent_pitching_context
        )
    )

    saber.update(pitching)

    saber_df = pd.DataFrame(
        [saber] * len(X),
        index=X.index
    )

    return pd.concat(
        [
            X.reset_index(drop=True),
            saber_df.reset_index(drop=True)
        ],
        axis=1
    )


# ============================================================
# 6. TIME-SERIES VALIDATION
# ============================================================

def evaluate_time_series_model(
    model,
    X,
    y,
    n_splits=5
):

    if len(X) < 12:
        return None, None

    n_splits = min(
        n_splits,
        max(
            2,
            len(X) // 8
        )
    )

    tscv = TimeSeriesSplit(
        n_splits=n_splits
    )

    predictions = []
    actuals = []

    for train_idx, test_idx in tscv.split(X):

        fold_model = clone(model)

        fold_model.fit(
            X.iloc[train_idx],
            y[train_idx]
        )

        pred = fold_model.predict(
            X.iloc[test_idx]
        )

        predictions.extend(
            pred.tolist()
        )

        actuals.extend(
            y[test_idx].tolist()
        )

    actuals = np.asarray(
        actuals
    )

    predictions = np.asarray(
        predictions
    )

    return {
        "MAE": mean_absolute_error(
            actuals,
            predictions
        ),

        "RMSE": np.sqrt(
            mean_squared_error(
                actuals,
                predictions
            )
        ),

        "R2": (
            r2_score(
                actuals,
                predictions
            )
            if len(np.unique(actuals)) > 1
            else np.nan
        )
    }, predictions


def train_best_model(
    X,
    y,
    model_names
):

    results = []
    fitted = {}

    for name in model_names:

        model = model_factory(
            name
        )

        metrics, _ = (
            evaluate_time_series_model(
                model,
                X,
                y
            )
        )

        if metrics:

            results.append({
                "Model": name,
                **metrics
            })

            final_model = clone(
                model
            )

            final_model.fit(
                X,
                y
            )

            fitted[name] = final_model

    result_df = pd.DataFrame(
        results
    )

    if not result_df.empty:

        best_name = (
            result_df
            .sort_values("RMSE")
            .iloc[0]["Model"]
        )

    else:

        best_name = model_names[0]

    return (
        result_df,
        fitted,
        best_name
    )


# ============================================================
# 7. ENSEMBLE
# ============================================================

def inverse_rmse_weights(
    forecast_df
):

    df = forecast_df.copy()

    valid = df[
        df["RMSE"].notna()
        & (df["RMSE"] > 0)
    ].copy()

    if valid.empty:

        df["Weight"] = (
            1.0 / len(df)
        )

        return df

    valid["RawWeight"] = (
        1.0
        / valid["RMSE"]
    )

    total = valid[
        "RawWeight"
    ].sum()

    valid["Weight"] = (
        valid["RawWeight"]
        / total
    )

    df["Weight"] = 0.0

    for idx in valid.index:

        df.loc[
            idx,
            "Weight"
        ] = valid.loc[
            idx,
            "Weight"
        ]

    return df


def build_ensemble_forecast(
    classical_df,
    ml_forecast,
    ml_rmse
):

    rows = []

    # Classical models
    for _, row in classical_df.iterrows():

        rows.append({
            "Method": row["Method"],
            "Forecast": row["Forecast"],
            "RMSE": row["RMSE"]
        })

    # ML
    rows.append({
        "Method": "ML",
        "Forecast": ml_forecast,
        "RMSE": ml_rmse
    })

    ensemble_df = pd.DataFrame(
        rows
    )

    ensemble_df = (
        inverse_rmse_weights(
            ensemble_df
        )
    )

    final_forecast = float(
        np.sum(
            ensemble_df["Forecast"]
            * ensemble_df["Weight"]
        )
    )

    return (
        ensemble_df,
        final_forecast
    )


# ============================================================
# 8. STREAMLIT APP
# ============================================================

st.set_page_config(
    page_title="MLB Match Predictor",
    layout="wide",
    page_icon="🏟️"
)

st.title(
    "🏟️MLB Maç Projeksiyon Motoru"
)

st.caption(
    "Hücum Formu + Starting Pitcher Formu + "
    "Bullpen Formu + WMA + EWMA(3) + Lineer Regresyon + "
    "Polinom Trend + AR(1) + ML + Ensemble + Monte Carlo"
)


# ============================================================
# SIDEBAR
# ============================================================

st.sidebar.header(
    "⚙️ Ayarlar"
)

models_list = [
    "Random Forest",
    "Gradient Boosting",
    "HistGradient Boosting",
    "Ridge Regression",
    "Poisson Regression"
]

if XGB_AVAILABLE:
    models_list.append(
        "XGBoost"
    )

model_choice = st.sidebar.selectbox(
    "Tercih Edilen Model",
    models_list
)

auto_select = st.sidebar.checkbox(
    "Walk-forward testinde en iyiyi otomatik seç",
    value=True
)

iters = st.sidebar.number_input(
    "Monte Carlo İterasyonu",
    1000,
    50000,
    10000,
    1000
)

park_factor = st.sidebar.number_input(
    "Park Factor (Stadyum Etkisi)",
    70.0,
    130.0,
    100.0,
    1.0
)

# ------------------------------------------------------------
# POSTSEASON DATABASE
# ------------------------------------------------------------
postseason_db = None
postseason_profiles = pd.DataFrame()

try:
    postseason_db = load_postseason_database(POSTSEASON_CSV)
    postseason_profiles = build_postseason_profiles(postseason_db)
except Exception as e:
    st.sidebar.warning(f"MaçML.csv okunamadı: {e}")

available_teams = (
    postseason_profiles["team"].tolist()
    if not postseason_profiles.empty
    else ["Braves", "Dodgers", "Yankees", "Phillies"]
)

st.sidebar.markdown("---")
st.sidebar.subheader("🏆 Postseason Geçmişi")

away_team = st.sidebar.selectbox(
    "Away Takımı",
    available_teams,
    index=available_teams.index("Braves") if "Braves" in available_teams else 0
)

home_default = (
    "Dodgers" if "Dodgers" in available_teams
    else available_teams[min(1, len(available_teams)-1)]
)
home_team = st.sidebar.selectbox(
    "Home Takımı",
    available_teams,
    index=available_teams.index(home_default)
)

if away_team == home_team:
    st.sidebar.warning("Away ve Home aynı takım seçildi.")

away_post_profile = get_postseason_profile(postseason_profiles, away_team)
home_post_profile = get_postseason_profile(postseason_profiles, home_team)

if away_post_profile and home_post_profile:
    st.sidebar.metric(
        "Postseason Index",
        f"{away_team}: {away_post_profile['postseason_index']:.1f}"
    )
    st.sidebar.metric(
        "Postseason Index ",
        f"{home_team}: {home_post_profile['postseason_index']:.1f}"
    )
    st.sidebar.caption(
        f"Matchup edge: "
        f"{away_post_profile['postseason_index'] - home_post_profile['postseason_index']:+.1f}"
    )


# ============================================================
# LEAGUE BASELINES
# ============================================================

league_stats = {

    "xwOBA": {
        "mean": 0.320,
        "std": 0.025
    },

    "wRC+": {
        "mean": 100.0,
        "std": 15.0
    },

    "OPS": {
        "mean": 0.720,
        "std": 0.080
    }
}


st.markdown("---")


# ============================================================
# MATCH INPUTS
# ============================================================

col_away, col_home = st.columns(2)


def generate_mock_runs():

    return ",".join(
        map(
            str,
            np.random.poisson(
                4.5,
                100
            )
        )
    )


# ============================================================
# AWAY
# ============================================================

with col_away:

    st.header(
        f"✈️ {away_team}"
    )

    away_runs_100 = st.text_area(
        "Son 100 Maç Skorları (Runs)",
        value=generate_mock_runs(),
        height=80
    )

    a_tab1, a_tab2, a_tab3 = st.tabs([
        "Son 30 Maç",
        "Son 7 Maç",
        "Son 5 (H2H)"
    ])

    with a_tab1:

        away_runs_30 = st.text_input(
            "Away Son 30",
            value=(
                "4,2,1,5,0,3,7,2,1,4,"
                "5,2,3,6,1,0,4,2,5,3,"
                "1,7,8,2,0,3,4,1,2,5"
            )
        )

    with a_tab2:

        away_runs_7 = st.text_input(
            "Away Son 7",
            value="3,4,1,2,5,0,4"
        )

    with a_tab3:

        away_runs_5 = st.text_input(
            "Away Son 5 (H2H)",
            value="2,4,3,1,5"
        )


    # ========================================================
    # AWAY OFFENSE
    # ========================================================

    st.markdown(
        "#### 🎯Hücum Formu"
    )

    df_away_saber = st.data_editor(

        pd.DataFrame(
            {
                "xwOBA": [
                    0.325,
                    0.330,
                    0.310
                ],

                "wRC+": [
                    105,
                    112,
                    95
                ],

                "OPS": [
                    0.735,
                    0.760,
                    0.700
                ]
            },

            index=[
                "season",
                "l30",
                "l7"
            ]
        ),

        use_container_width=True,

        key="saber_away"
    )


    # ========================================================
    # AWAY OPPONENT DEFENSE
    # ========================================================

    st.markdown(
        "#### 🛡️Savunma Formu (Ev)"
    )

    st.markdown(
        "**Starting Pitcher**"
    )

    df_away_sp = st.data_editor(

        pd.DataFrame(
            {
                "ERA": [
                    3.80,
                    3.65,
                    3.90
                ],

                "FIP": [
                    3.75,
                    3.60,
                    3.95
                ]
            },

            index=[
                "season",
                "l30",
                "l7"
            ]
        ),

        use_container_width=True,

        key="sp_away_opponent"
    )


    st.markdown(
        "**Bullpen**"
    )

    df_away_bp = st.data_editor(

        pd.DataFrame(
            {
                "ERA": [
                    4.10,
                    3.95,
                    4.25
                ],

                "FIP": [
                    4.00,
                    3.85,
                    4.15
                ]
            },

            index=[
                "season",
                "l30",
                "l7"
            ]
        ),

        use_container_width=True,

        key="bp_away_opponent"
    )


# ============================================================
# HOME
# ============================================================

with col_home:

    st.header(
        f"🏠 {home_team}"
    )

    home_runs_100 = st.text_area(
        "Son 100 Maç Skorları (Runs)",
        value=generate_mock_runs(),
        height=80
    )

    h_tab1, h_tab2, h_tab3 = st.tabs([
        "Son 30 Maç",
        "Son 7 Maç",
        "Son 5 (H2H)"
    ])

    with h_tab1:

        home_runs_30 = st.text_input(
            "Home Son 30",
            value=(
                "3,5,2,4,1,6,2,3,4,1,"
                "0,5,6,2,3,4,1,2,5,7,"
                "3,2,1,4,5,6,3,2,1,4"
            )
        )

    with h_tab2:

        home_runs_7 = st.text_input(
            "Home Son 7",
            value="5,6,3,2,1,4,3"
        )

    with h_tab3:

        home_runs_5 = st.text_input(
            "Home Son 5 (H2H)",
            value="4,1,2,6,3"
        )


    # ========================================================
    # HOME OFFENSE
    # ========================================================

    st.markdown(
        "#### 🎯Hücum Formu"
    )

    df_home_saber = st.data_editor(

        pd.DataFrame(
            {
                "xwOBA": [
                    0.315,
                    0.305,
                    0.340
                ],

                "wRC+": [
                    98,
                    95,
                    115
                ],

                "OPS": [
                    0.710,
                    0.690,
                    0.770
                ]
            },

            index=[
                "season",
                "l30",
                "l7"
            ]
        ),

        use_container_width=True,

        key="saber_home"
    )


    # ========================================================
    # HOME OPPONENT DEFENSE
    # ========================================================

    st.markdown(
        "#### 🛡️Savunma Formu (Dep)"
    )

    st.markdown(
        "**Starting Pitcher**"
    )

    df_home_sp = st.data_editor(

        pd.DataFrame(
            {
                "ERA": [
                    4.20,
                    4.05,
                    4.35
                ],

                "FIP": [
                    4.05,
                    3.95,
                    4.40
                ]
            },

            index=[
                "season",
                "l30",
                "l7"
            ]
        ),

        use_container_width=True,

        key="sp_home_opponent"
    )


    st.markdown(
        "**Bullpen**"
    )

    df_home_bp = st.data_editor(

        pd.DataFrame(
            {
                "ERA": [
                    3.90,
                    3.75,
                    4.10
                ],

                "FIP": [
                    3.85,
                    3.70,
                    4.05
                ]
            },

            index=[
                "season",
                "l30",
                "l7"
            ]
        ),

        use_container_width=True,

        key="bp_home_opponent"
    )


# ============================================================
# PREDICTION
# ============================================================

st.markdown("---")


if st.button(
    "🚀Maçı Simüle Et",
    use_container_width=True
):

    with st.spinner(
        "Forecasting modelleri, ML ve Monte Carlo çalıştırılıyor..."
    ):

        try:

            # =================================================
            # PARSE
            # =================================================

            a_100 = parse_series(
                away_runs_100,
                31,
                100
            )

            h_100 = parse_series(
                home_runs_100,
                31,
                100
            )


            # =================================================
            # ML TRAINING DATA
            # =================================================

            X_train_away, y_away = (
                build_training_rows_from_100_game_logs(
                    a_100,
                    30
                )
            )

            X_train_home, y_home = (
                build_training_rows_from_100_game_logs(
                    h_100,
                    30
                )
            )


            # =================================================
            # PITCHING CONTEXT
            #
            # IMPORTANT:
            #
            # Away score →
            # Home SP + Home Bullpen
            #
            # Home score →
            # Away SP + Away Bullpen
            # =================================================

            away_context = {

                "sp_era_season":
                    df_away_sp.loc[
                        "season",
                        "ERA"
                    ],

                "sp_era_l30":
                    df_away_sp.loc[
                        "l30",
                        "ERA"
                    ],

                "sp_era_l7":
                    df_away_sp.loc[
                        "l7",
                        "ERA"
                    ],

                "sp_fip_season":
                    df_away_sp.loc[
                        "season",
                        "FIP"
                    ],

                "sp_fip_l30":
                    df_away_sp.loc[
                        "l30",
                        "FIP"
                    ],

                "sp_fip_l7":
                    df_away_sp.loc[
                        "l7",
                        "FIP"
                    ],

                "bp_era_season":
                    df_away_bp.loc[
                        "season",
                        "ERA"
                    ],

                "bp_era_l30":
                    df_away_bp.loc[
                        "l30",
                        "ERA"
                    ],

                "bp_era_l7":
                    df_away_bp.loc[
                        "l7",
                        "ERA"
                    ],

                "bp_fip_season":
                    df_away_bp.loc[
                        "season",
                        "FIP"
                    ],

                "bp_fip_l30":
                    df_away_bp.loc[
                        "l30",
                        "FIP"
                    ],

                "bp_fip_l7":
                    df_away_bp.loc[
                        "l7",
                        "FIP"
                    ],

                "home_advantage": 0,

                "park_factor":
                    park_factor
            }


            home_context = {

                "sp_era_season":
                    df_home_sp.loc[
                        "season",
                        "ERA"
                    ],

                "sp_era_l30":
                    df_home_sp.loc[
                        "l30",
                        "ERA"
                    ],

                "sp_era_l7":
                    df_home_sp.loc[
                        "l7",
                        "ERA"
                    ],

                "sp_fip_season":
                    df_home_sp.loc[
                        "season",
                        "FIP"
                    ],

                "sp_fip_l30":
                    df_home_sp.loc[
                        "l30",
                        "FIP"
                    ],

                "sp_fip_l7":
                    df_home_sp.loc[
                        "l7",
                        "FIP"
                    ],

                "bp_era_season":
                    df_home_bp.loc[
                        "season",
                        "ERA"
                    ],

                "bp_era_l30":
                    df_home_bp.loc[
                        "l30",
                        "ERA"
                    ],

                "bp_era_l7":
                    df_home_bp.loc[
                        "l7",
                        "ERA"
                    ],

                "bp_fip_season":
                    df_home_bp.loc[
                        "season",
                        "FIP"
                    ],

                "bp_fip_l30":
                    df_home_bp.loc[
                        "l30",
                        "FIP"
                    ],

                "bp_fip_l7":
                    df_home_bp.loc[
                        "l7",
                        "FIP"
                    ],

                "home_advantage": 1,

                "park_factor":
                    park_factor
            }


            # =================================================
            # AUGMENT ML DATA
            # =================================================

            X_train_away = augment_training_data(
                X_train_away,
                df_away_saber.to_dict(),
                league_stats,
                away_context
            )

            X_train_home = augment_training_data(
                X_train_home,
                df_home_saber.to_dict(),
                league_stats,
                home_context
            )

            # =================================================
            # POSTSEASON FEATURES
            # =================================================
            # Her takımın CSV'deki 2000+ postseason geçmişi ML
            # feature setine eklenir.
            X_train_away = add_postseason_ml_features(
                X_train_away,
                away_post_profile,
                home_post_profile
            )

            X_train_home = add_postseason_ml_features(
                X_train_home,
                home_post_profile,
                away_post_profile
            )


            # =================================================
            # PREDICTION VECTOR
            # =================================================

            def build_pred_vector(
                runs_30_txt,
                runs_7_txt,
                runs_5_txt,
                saber_dict,
                context,
                X_train_ref
            ):

                r30 = parse_series(
                    runs_30_txt
                )

                r7 = parse_series(
                    runs_7_txt
                )

                r5 = parse_series(
                    runs_5_txt
                )

                pred_feats = (
                    MLFeatureExtractor
                    .extract_time_series_features(
                        r30,
                        prefix="runs"
                    )
                )

                pred_feats.update({

                    "runs_l7_mean":
                        float(
                            np.mean(r7)
                        ),

                    "runs_l5_h2h_mean":
                        float(
                            np.mean(r5)
                        ),

                    "runs_l5_h2h_max":
                        float(
                            np.max(r5)
                        )
                })

                pred_feats.update(
                    MLFeatureExtractor
                    .extract_sabermetric_features(
                        saber_dict,
                        league_stats
                    )
                )

                pred_feats.update(
                    MLFeatureExtractor
                    .add_pitching_context(
                        {},
                        context
                    )
                )

                return (
                    pd.DataFrame(
                        [pred_feats]
                    )
                    .reindex(
                        columns=X_train_ref.columns,
                        fill_value=0.0
                    )
                )


            X_pred_away = build_pred_vector(
                away_runs_30,
                away_runs_7,
                away_runs_5,
                df_away_saber.to_dict(),
                away_context,
                X_train_away
            )

            X_pred_home = build_pred_vector(
                home_runs_30,
                home_runs_7,
                home_runs_5,
                df_home_saber.to_dict(),
                home_context,
                X_train_home
            )

            X_pred_away = add_postseason_ml_features(
                X_pred_away,
                away_post_profile,
                home_post_profile
            ).reindex(columns=X_train_away.columns, fill_value=0.0)

            X_pred_home = add_postseason_ml_features(
                X_pred_home,
                home_post_profile,
                away_post_profile
            ).reindex(columns=X_train_home.columns, fill_value=0.0)


            # =================================================
            # ML MODELS
            # =================================================

            res_away, models_away, best_away = (
                train_best_model(
                    X_train_away,
                    y_away,
                    models_list
                )
            )

            res_home, models_home, best_home = (
                train_best_model(
                    X_train_home,
                    y_home,
                    models_list
                )
            )


            sel_away = (
                best_away
                if auto_select
                else model_choice
            )

            sel_home = (
                best_home
                if auto_select
                else model_choice
            )


            model_a = models_away.get(
                sel_away,
                models_away[best_away]
            )

            model_h = models_home.get(
                sel_home,
                models_home[best_home]
            )


            # =================================================
            # RAW ML FORECAST
            # =================================================

            ml_lambda_away = max(
                0.01,
                float(
                    model_a.predict(
                        X_pred_away
                    )[0]
                )
            )

            ml_lambda_home = max(
                0.01,
                float(
                    model_h.predict(
                        X_pred_home
                    )[0]
                )
            )

            # =================================================
            # POSTSEASON WEIGHTED ML FORECAST
            # =================================================
            ml_lambda_away_raw = ml_lambda_away
            ml_lambda_home_raw = ml_lambda_home

            ml_lambda_away, postseason_mult_away = (
                apply_postseason_weight(
                    ml_lambda_away_raw,
                    away_post_profile,
                    home_post_profile
                )
            )

            ml_lambda_home, postseason_mult_home = (
                apply_postseason_weight(
                    ml_lambda_home_raw,
                    home_post_profile,
                    away_post_profile
                )
            )


            # =================================================
            # ML RMSE
            # =================================================

            ml_rmse_away = float(
                res_away.loc[
                    res_away["Model"] == sel_away,
                    "RMSE"
                ].iloc[0]
            )

            ml_rmse_home = float(
                res_home.loc[
                    res_home["Model"] == sel_home,
                    "RMSE"
                ].iloc[0]
            )


            # =================================================
            # CLASSICAL FORECASTING
            # =================================================

            classical_away = (
                classical_forecast_summary(
                    a_100
                )
            )

            classical_home = (
                classical_forecast_summary(
                    h_100
                )
            )


            # =================================================
            # ENSEMBLE
            # =================================================

            ensemble_away, lambda_away = (
                build_ensemble_forecast(
                    classical_away,
                    ml_lambda_away,
                    ml_rmse_away
                )
            )

            ensemble_home, lambda_home = (
                build_ensemble_forecast(
                    classical_home,
                    ml_lambda_home,
                    ml_rmse_home
                )
            )


            lambda_away = max(
                0.01,
                lambda_away
            )

            lambda_home = max(
                0.01,
                lambda_home
            )


            # =================================================
            # MONTE CARLO
            # =================================================

            sim_away = np.random.poisson(
                lambda_away,
                iters
            )

            sim_home = np.random.poisson(
                lambda_home,
                iters
            )

            sim_total = (
                sim_away
                + sim_home
            )


            # =================================================
            # PROBABILITIES
            # =================================================

            away_win_prob = (
                np.mean(
                    sim_away > sim_home
                ) * 100
            )

            home_win_prob = (
                np.mean(
                    sim_home > sim_away
                ) * 100
            )

            tie_prob = (
                np.mean(
                    sim_away == sim_home
                ) * 100
            )

            over_8_5 = (
                np.mean(
                    sim_total > 8.5
                ) * 100
            )


            # =================================================
            # OUTPUT
            # =================================================

            st.success(
                "Forecasting, Walk-Forward Validation, "
                "Ensemble ve Monte Carlo tamamlandı!"
            )


            # =================================================
            # MAIN METRICS
            # =================================================

            c1, c2, c3, c4 = st.columns(4)

            c1.metric(
                "✈️ Away Final Expected Runs",
                f"{lambda_away:.2f}"
            )

            c2.metric(
                "🏠 Home Final Expected Runs",
                f"{lambda_home:.2f}"
            )

            c3.metric(
                "🔥 Toplam Beklenen Runs",
                f"{lambda_away + lambda_home:.2f}"
            )

            c4.metric(
                "📈 8.5 Üst Olasılığı",
                f"%{over_8_5:.1f}"
            )

            st.info(
                f"🏆 **Postseason etkisi:** "
                f"{away_team} ML × {postseason_mult_away:.3f} → "
                f"{ml_lambda_away:.2f} | "
                f"{home_team} ML × {postseason_mult_home:.3f} → "
                f"{ml_lambda_home:.2f}"
            )


            # =================================================
            # FORECAST BREAKDOWN
            # =================================================

            st.markdown("---")

            st.subheader(
                "🔬 Tahmin Motorlarının Ayrışması"
            )

            fc1, fc2 = st.columns(2)


            with fc1:

                st.markdown(
                    "### ✈️ Away"
                )

                display_away = (
                    ensemble_away[
                        [
                            "Method",
                            "Forecast",
                            "RMSE",
                            "Weight"
                        ]
                    ].copy()
                )

                st.dataframe(
                    display_away.style.format({
                        "Forecast": "{:.3f}",
                        "RMSE": "{:.3f}",
                        "Weight": "{:.2%}"
                    }),
                    hide_index=True,
                    use_container_width=True
                )


            with fc2:

                st.markdown(
                    "### 🏠 Home"
                )

                display_home = (
                    ensemble_home[
                        [
                            "Method",
                            "Forecast",
                            "RMSE",
                            "Weight"
                        ]
                    ].copy()
                )

                st.dataframe(
                    display_home.style.format({
                        "Forecast": "{:.3f}",
                        "RMSE": "{:.3f}",
                        "Weight": "{:.2%}"
                    }),
                    hide_index=True,
                    use_container_width=True
                )


            # =================================================
            # FORECAST COMPARISON GRAPH
            # =================================================

            st.markdown("---")

            st.subheader(
                "📈 Forecasting Modelleri Karşılaştırması"
            )

            graph_df = pd.DataFrame({
                "Method": ensemble_away["Method"].tolist() + ["Ensemble"],
                f"{away_team} Forecast": ensemble_away["Forecast"].tolist() + [lambda_away],
                f"{home_team} Forecast": ensemble_home["Forecast"].tolist() + [lambda_home]
            })

            fig_forecast = go.Figure()

            fig_forecast.add_trace(
                go.Bar(
                    x=graph_df["Method"],
                    y=graph_df[f"{away_team} Forecast"],
                    name=away_team
                )
            )

            fig_forecast.add_trace(
                go.Bar(
                    x=graph_df["Method"],
                    y=graph_df[f"{home_team} Forecast"],
                    name=home_team
                )
            )

            fig_forecast.update_layout(
                barmode="group",
                xaxis_title="Forecasting Modeli",
                yaxis_title="Beklenen Runs",
                title="WMA / EWMA(3) / Lineer Regresyon / Polinom Trend / AR(1) / ML / Ensemble"
            )

            st.plotly_chart(
                fig_forecast,
                use_container_width=True
            )


            # =================================================
            # MODEL EXPLANATION
            # =================================================

            st.markdown("---")

            st.subheader(
                "🧠 Final Ensemble Yapısı"
            )

            st.write(
                f"""
                **Away:**
                WMA + EWMA(3) + Lineer Regresyon + Polinom Trend + AR(1) + ML → **{lambda_away:.2f}**

                **Home:**
                WMA + EWMA(3) + Lineer Regresyon + Polinom Trend + AR(1) + ML → **{lambda_home:.2f}**

                Ensemble ağırlıkları geçmiş backtest RMSE
                değerlerinin tersine göre belirlenmiştir.
                Daha düşük RMSE → daha yüksek ağırlık.
                """
            )


            # =================================================
            # WIN PROBABILITIES
            # =================================================

            st.markdown(
                "### 🏆 Kazanma Olasılıkları"
            )

            st.progress(
                away_win_prob / 100
            )

            st.write(
                f"""
                **Deplasman:** %{away_win_prob:.1f}
                **Beraberlik/Uzatma:** %{tie_prob:.1f}
                **Ev Sahibi:** %{home_win_prob:.1f}
                """
            )


            # =================================================
            # MONTE CARLO DISTRIBUTIONS
            # =================================================

            col_chart1, col_chart2 = st.columns(2)


            with col_chart1:

                df_dist = pd.DataFrame({

                    "Score":
                        np.concatenate([
                            sim_away,
                            sim_home
                        ]),

                    "Team":
                        (
                            ["Away"] * iters
                            + ["Home"] * iters
                        )
                })

                fig1 = px.histogram(
                    df_dist,
                    x="Score",
                    color="Team",
                    barmode="group",
                    title=(
                        "Takım Skor Dağılımı "
                        "(Monte Carlo)"
                    ),
                    histnorm="percent"
                )

                st.plotly_chart(
                    fig1,
                    use_container_width=True
                )


            with col_chart2:

                fig2 = px.histogram(
                    sim_total,
                    title=(
                        "Toplam Maç Skoru "
                        "Dağılımı"
                    ),
                    labels={
                        "value":
                            "Toplam Sayı"
                    },
                    histnorm="percent"
                )

                fig2.add_vline(
                    x=8.5,
                    line_dash="dash",
                    annotation_text="8.5 Line"
                )

                st.plotly_chart(
                    fig2,
                    use_container_width=True
                )


            # =================================================
            # POSTSEASON DATABASE OUTPUT
            # =================================================

            st.markdown("---")
            st.subheader("🏆 CSV Postseason Etkisi")

            if away_post_profile and home_post_profile:
                ps_out = pd.DataFrame([
                    {
                        "Takım": away_team,
                        "Postseason Index": away_post_profile["postseason_index"],
                        "Postseason Sezonu": away_post_profile["postseason_seasons"],
                        "Şampiyonluk": away_post_profile["titles"],
                        "PS ERA": away_post_profile["postseason_era"],
                        "PS FIP": away_post_profile["postseason_fip"]
                    },
                    {
                        "Takım": home_team,
                        "Postseason Index": home_post_profile["postseason_index"],
                        "Postseason Sezonu": home_post_profile["postseason_seasons"],
                        "Şampiyonluk": home_post_profile["titles"],
                        "PS ERA": home_post_profile["postseason_era"],
                        "PS FIP": home_post_profile["postseason_fip"]
                    }
                ])

                st.dataframe(
                    ps_out.style.format({
                        "Postseason Index": "{:.2f}",
                        "PS ERA": "{:.2f}",
                        "PS FIP": "{:.2f}"
                    }),
                    hide_index=True,
                    use_container_width=True
                )

                st.caption(
                    f"CSV etkisi ML feature setinde "
                    f"`postseason_index` ve `postseason_matchup_edge` "
                    f"olarak yer alır; ayrıca final ML lambda'ya "
                    f"maksimum ±%{POSTSEASON_WEIGHT*100:.0f} bounded adjustment uygulanır."
                )

            # =================================================
            # VALIDATION
            # =================================================

            st.markdown("---")

            st.subheader(
                "📊 ML Model Validation Performansı"
            )

            v1, v2 = st.columns(2)


            with v1:

                st.write(
                    f"**Away Seçilen Model:** "
                    f"{sel_away}"
                )

                st.dataframe(
                    res_away.style.format({
                        "MAE": "{:.3f}",
                        "RMSE": "{:.3f}",
                        "R2": "{:.3f}"
                    }),
                    hide_index=True,
                    use_container_width=True
                )


            with v2:

                st.write(
                    f"**Home Seçilen Model:** "
                    f"{sel_home}"
                )

                st.dataframe(
                    res_home.style.format({
                        "MAE": "{:.3f}",
                        "RMSE": "{:.3f}",
                        "R2": "{:.3f}"
                    }),
                    hide_index=True,
                    use_container_width=True
                )


            # =================================================
            # CLASSICAL MODEL VALIDATION
            # =================================================

            st.markdown("---")

            st.subheader(
                "📈 Klasik Forecasting Backtest"
            )

            classical_display_away = (
                classical_away.copy()
            )

            classical_display_home = (
                classical_home.copy()
            )


            c1, c2 = st.columns(2)

            with c1:

                st.write(
                    "**Away Classical Forecasting**"
                )

                st.dataframe(
                    classical_display_away.style.format({
                        "Forecast": "{:.3f}",
                        "RMSE": "{:.3f}"
                    }),
                    hide_index=True,
                    use_container_width=True
                )


            with c2:

                st.write(
                    "**Home Classical Forecasting**"
                )

                st.dataframe(
                    classical_display_home.style.format({
                        "Forecast": "{:.3f}",
                        "RMSE": "{:.3f}"
                    }),
                    hide_index=True,
                    use_container_width=True
                )


            # =================================================
            # SHAP
            # =================================================

            st.markdown("---")

            st.subheader(
                "🔎 Feature Importance & "
                "Model Karar Mekanizması"
            )


            if (
                SHAP_AVAILABLE
                and hasattr(
                    model_a,
                    "feature_importances_"
                )
            ):

                st.write(
                    "Deplasman modeli için SHAP"
                )

                try:

                    explainer = (
                        shap.TreeExplainer(
                            model_a
                        )
                    )

                    shap_values = (
                        explainer.shap_values(
                            X_train_away
                        )
                    )

                    fig_shap, ax = plt.subplots(
                        figsize=(8, 4)
                    )

                    shap.summary_plot(
                        shap_values,
                        X_train_away,
                        max_display=12,
                        show=False
                    )

                    st.pyplot(
                        fig_shap
                    )

                except Exception:

                    st.warning(
                        "Ağaç tabanlı SHAP çizilemedi."
                    )


            elif hasattr(
                model_a,
                "feature_importances_"
            ):

                fi = pd.DataFrame({

                    "Feature":
                        X_train_away.columns,

                    "Importance":
                        model_a.feature_importances_

                }).sort_values(
                    "Importance",
                    ascending=False
                )

                fig_fi = px.bar(
                    fi.head(12)
                    .sort_values(
                        "Importance"
                    ),

                    x="Importance",
                    y="Feature",
                    orientation="h",

                    title=(
                        "Top 12 Etkili Faktör "
                        "(Deplasman)"
                    )
                )

                st.plotly_chart(
                    fig_fi,
                    use_container_width=True
                )


        except Exception as e:

            st.error(
                f"Tahmin işlemi sırasında hata: {e}"
            )

            st.exception(e)

In [3]:
import urllib
print("Şifre:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

Şifre: 34.81.135.244


In [4]:
import subprocess
import os
import urllib.request
import platform
import re
import time
import threading

system = platform.system()
binary_name = "cloudflared.exe" if system == "Windows" else "cloudflared"

if not os.path.exists(binary_name):
    print("Cloudflare tünel aracı indiriliyor")
    if system == "Windows":
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-windows-amd64.exe"
    else:
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    urllib.request.urlretrieve(url, binary_name)
    if system != "Windows":
        os.chmod(binary_name, 0o755)
    print("İndirme tamamlandı!")

def run_streamlit():
    subprocess.run(["streamlit", "run", "sabermetrics_gui.py", "--server.headless=true"])

streamlit_thread = threading.Thread(target=run_streamlit, daemon=True)
streamlit_thread.start()
time.sleep(3)

print("Cloudflare tüneli açılıyor")
cmd = [os.path.abspath(binary_name), "tunnel", "--url", "http://localhost:8501"]
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

tunnel_url = None
for _ in range(15):
    line = process.stderr.readline()
    match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(1)

if tunnel_url:
    print(f"\n==================================================")
    print(f"BAĞLANTI HAZIR!")
    print(f"{tunnel_url}")
    print(f"==================================================\n")
else:
    print("Tünel linki otomatik yakalanamadı")

Cloudflare tünel aracı indiriliyor
İndirme tamamlandı!
Cloudflare tüneli açılıyor

BAĞLANTI HAZIR!
https://divisions-material-captain-sale.trycloudflare.com

